# EX_00 — PyTorch vs TensorFlow (ejercicios)

**Notebook de referencia:** `notebook/00_Frameworks_Pytorch_vs_Tensorflow.ipynb`

**Tiempo orientativo:** ~30 minutos.

En esta hoja practicarás ideas equivalentes en ambos frameworks: tensores, capas lineales y un forward pass mínimo.


## Actividad 1 — Activación a mano

Implementa en NumPy una función `relu` y otra `sigmoid` y comprueba que coinciden con `torch` y `tensorflow` en un vector de prueba.

*Hint:* use `torch.relu`, `tf.nn.relu`; for sigmoid use `torch.sigmoid` and `tf.nn.sigmoid`.


In [1]:
import numpy as np
import torch
import tensorflow as tf

# 1. Implementación de las funciones de activación en NumPy
def relu_np(z):
    # ReLU devuelve el máximo entre 0 y el valor de z
    return np.maximum(0, z)

def sigmoid_np(z):
    # La fórmula matemática de la sigmoide es: 1 / (1 + e^(-z))
    return 1 / (1 + np.exp(-z))

# Vector de prueba proporcionado
x = np.array([-2.0, 0.0, 1.5], dtype=np.float32)

# 2. Calcular las salidas con tus funciones de NumPy
relu_out_np = relu_np(x)
sigmoid_out_np = sigmoid_np(x)

# 3. Calcular las salidas con PyTorch
# (Convertimos el array de NumPy a tensor de PyTorch)
x_torch = torch.from_numpy(x)
relu_out_torch = torch.relu(x_torch).numpy()
sigmoid_out_torch = torch.sigmoid(x_torch).numpy()

# 4. Calcular las salidas con TensorFlow
# (Convertimos el array de NumPy a tensor de TensorFlow)
x_tf = tf.convert_to_tensor(x)
relu_out_tf = tf.nn.relu(x_tf).numpy()
sigmoid_out_tf = tf.nn.sigmoid(x_tf).numpy()

# 5. ASSERTS: Comprobar que todos los resultados son idénticos
# Usamos rtol/atol por precisión decimal de floats
print("Verificando ReLU...")
np.testing.assert_allclose(relu_out_np, relu_out_torch, rtol=1e-5, atol=1e-5)
np.testing.assert_allclose(relu_out_np, relu_out_tf, rtol=1e-5, atol=1e-5)
print("¡ReLU coincide perfectamente en NumPy, PyTorch y TensorFlow!")

print("\nVerificando Sigmoid...")
np.testing.assert_allclose(sigmoid_out_np, sigmoid_out_torch, rtol=1e-5, atol=1e-5)
np.testing.assert_allclose(sigmoid_out_np, sigmoid_out_tf, rtol=1e-5, atol=1e-5)
print("¡Sigmoid coincide perfectamente en NumPy, PyTorch y TensorFlow!")

# Imprimimos los resultados para que los veas en pantalla
print("\n--- Resultados finales ---")
print(f"Entrada x:      {x}")
print(f"Salida ReLU:    {relu_out_np}")
print(f"Salida Sigmoid: {sigmoid_out_np}")


Verificando ReLU...
¡ReLU coincide perfectamente en NumPy, PyTorch y TensorFlow!

Verificando Sigmoid...
¡Sigmoid coincide perfectamente en NumPy, PyTorch y TensorFlow!

--- Resultados finales ---
Entrada x:      [-2.   0.   1.5]
Salida ReLU:    [0.  0.  1.5]
Salida Sigmoid: [0.11920293 0.5        0.8175745 ]


## Actividad 2 — Misma arquitectura, dos APIs

Define una red `Linear(10, 3)` + `ReLU` + `Linear(3, 1)` en **PyTorch** (`nn.Sequential`) y la misma en **Keras** (`Sequential`).

*Hint:* set seeds (`torch.manual_seed`, `tf.random.set_seed`) and use explicit init if you want to compare weights.


In [2]:
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Fijar semillas para asegurar reproducibilidad
torch.manual_seed(42)
tf.random.set_seed(42)

# Crear un input aleatorio común (batch=4, features=10)
# Usamos NumPy para que sea exactamente la misma entrada para ambos frameworks
x_np = np.random.randn(4, 10).astype(np.float32)

# ==========================================
# 2. DEFINICIÓN E INICIALIZACIÓN EN PYTORCH
# ==========================================
torch_model = nn.Sequential(
    nn.Linear(10, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
)

# Inicialización explícita para poder comparar resultados exactos
with torch.no_grad():
    # Inicializamos pesos a 0.5 y sesgos a 0.0 para probar
    torch_model[0].weight.fill_(0.5)
    torch_model[0].bias.fill_(0.0)
    torch_model[2].weight.fill_(0.5)
    torch_model[2].bias.fill_(0.0)

# Pasar el input a tensor de PyTorch y hacer el forward pass
x_torch = torch.from_numpy(x_np)
torch_output = torch_model(x_torch).detach().numpy()

# ==========================================
# 3. DEFINICIÓN E INICIALIZACIÓN EN KERAS
# ==========================================
keras_model = keras.Sequential([
    # Especificamos input_shape=(10,) para que construya los pesos de inmediato
    layers.Dense(3, activation='relu', input_shape=(10,),
                 kernel_initializer=tf.keras.initializers.Constant(0.5),
                 bias_initializer='zeros'),
    layers.Dense(1,
                 kernel_initializer=tf.keras.initializers.Constant(0.5),
                 bias_initializer='zeros')
])

# Pasar el input a tensor de TensorFlow y hacer el forward pass
x_tf = tf.convert_to_tensor(x_np)
keras_output = keras_model(x_tf).numpy()

# ==========================================
# 4. COMPROBACIÓN Y VERIFICACIÓN
# ==========================================
print("--- ENTRADA DE PRUEBA (BATCH=4, FEATURES=10) ---")
print(x_np)

print("\n--- SALIDA PYTORCH ---")
print(torch_output)

print("\n--- SALIDA KERAS ---")
print(keras_output)

# Validamos que las diferencias sean prácticamente cero
print("\nVerificando consistencia entre APIs...")
np.testing.assert_allclose(torch_output, keras_output, rtol=1e-5, atol=1e-5)
print("¡Éxito! Ambas arquitecturas devuelven salidas idénticas.")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


--- ENTRADA DE PRUEBA (BATCH=4, FEATURES=10) ---
[[ 0.19059521  1.1423233   0.12344468  1.7247235   0.9266151   0.38112432
  -0.22414133  0.28728604 -0.3759228   0.8105558 ]
 [-2.8471856  -0.13572216  0.5444613   0.69437516 -0.2392491   1.0305291
  -0.02977514 -0.26317695  1.1419836   1.0400321 ]
 [ 1.0909904   0.7346301  -1.1229056   0.33891413  0.74322695 -0.52210355
  -0.23271681 -0.13946936  0.43618318 -1.4896947 ]
 [ 0.357841   -0.36513117  1.7645935  -0.368839   -0.1077478  -0.0883644
  -2.1788058  -1.3414506   0.98507005  1.5062963 ]]

--- SALIDA PYTORCH ---
[[3.739953  ]
 [0.70220435]
 [0.        ]
 [0.12259653]]

--- SALIDA KERAS ---
[[3.7399528 ]
 [0.70220435]
 [0.        ]
 [0.12259658]]

Verificando consistencia entre APIs...
¡Éxito! Ambas arquitecturas devuelven salidas idénticas.


## Actividad 3 — Entrenamiento en mini-batch (conceptual + código corto)

Escribe un bucle de **una época** que: (1) muestree un batch sintético `X, y` para regresión, (2) calcule `MSE`, (3) haga `backward` / `gradient` y un paso de optimizador.

Elige **solo uno** de los dos frameworks para el bucle completo; en el otro, documenta en un comentario qué API usarías (`loss.backward`, `tape.gradient`, etc.).


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# =====================================================================
# INTERFAZ EQUIVALENTE EN TENSORFLOW / KERAS (DOCUMENTACIÓN CON COMENTARIOS)
# =====================================================================
# Si utilizáramos TensorFlow/Keras para esta actividad, la API clave sería:
#
# with tf.GradientTape() as tape:
#     predictions = keras_model(X)
#     loss = tf.keras.losses.MSE(y, predictions)
#
# gradients = tape.gradient(loss, keras_model.trainable_variables)
# optimizer.apply_gradients(zip(gradients, keras_model.trainable_variables))
# =====================================================================

# 1. Configuración del escenario en PyTorch
torch.manual_seed(42)

# Definimos un modelo lineal simple para la regresión: Input(10) -> Output(1)
model = nn.Linear(10, 1)

# Definimos la función de pérdida (MSE) y el optimizador (SGD)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 2. (1) Muestrear un batch sintético de datos X, y
# Supongamos un tamaño de batch = 8 y 10 características de entrada
X_batch = torch.randn(8, 10)
y_batch = torch.randn(8, 1)  # Valores objetivo reales (targets)

print("--- Estado inicial antes del paso de entrenamiento ---")
# Hacemos una predicción inicial antes de entrenar
initial_predictions = model(X_batch)
initial_loss = criterion(initial_predictions, y_batch)
print(f"Pérdida (MSE) inicial: {initial_loss.item():.4f}")

# 3. BUCLE DE UNA ÉPOCA (Paso de optimización)

# Pasamos el modelo a modo entrenamiento
model.train()

# a) Limpiar los gradientes acumulados del optimizador
optimizer.zero_grad()

# b) Forward pass: Calcular las predicciones del modelo
predictions = model(X_batch)

# c) (2) Calcular la función de pérdida (MSE)
loss = criterion(predictions, y_batch)

# d) (3) Backward pass: Calcular los gradientes con respecto a la pérdida
loss.backward()

# e) Realizar el paso del optimizador para actualizar los pesos
optimizer.step()

print("\n--- Estado final después del paso de entrenamiento ---")
# Volvemos a calcular la predicciones para comprobar si ha aprendido
with torch.no_grad():
    new_predictions = model(X_batch)
    new_loss = criterion(new_predictions, y_batch)
print(f"Pérdida (MSE) final:   {new_loss.item():.4f}")
print(f"¿La pérdida ha disminuido?: {'Sí, el optimizador funciona.' if new_loss < initial_loss else 'No'}")


--- Estado inicial antes del paso de entrenamiento ---
Pérdida (MSE) inicial: 2.1104

--- Estado final después del paso de entrenamiento ---
Pérdida (MSE) final:   2.0258
¿La pérdida ha disminuido?: Sí, el optimizador funciona.
